# **SUPPORT 3D Pipeline**

## **Initialization**

In [1]:
%pip install ipython
from IPython.display import clear_output
clear_output()
import shutil
import sys
import os
from pathlib import Path
from glob import glob
sys.path.append(os.path.expanduser('~/CloudCode/Models/SUPPORT').replace('home', 'data'))

import random
import logging
import time
import numpy as np
import torch
import skimage.io as skio
from pathlib import Path
import tifffile
import yaml

from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from types import SimpleNamespace
from model.SUPPORT import SUPPORT
from utils.dataset import gen_train_dataloader, random_transform
from utils.util import parse_arguments
from utils.train import train
from utils.dataset import DatasetSUPPORT_test_stitch
torch.cuda.empty_cache()

def ascii_plot(values, width=60, height=15):
    """Plot values as ASCII scatter plot with auto-scaling y-axis."""
    if len(values) == 0 or np.any(np.isnan(values)):
        return
    
    y_min, y_max = min(values), max(values)
    y_range = y_max - y_min if y_max > y_min else 0.001
    
    # Add small padding to y range
    y_min -= y_range * 0.05
    y_max += y_range * 0.05
    y_range = y_max - y_min
    
    # Create empty grid
    grid = [[' ' for _ in range(width)] for _ in range(height)]
    
    # Plot points
    for i, val in enumerate(values):
        x = int(i * (width - 1) / max(len(values) - 1, 1))
        y = int((val - y_min) / y_range * (height - 1))
        y = height - 1 - y  # Flip y-axis
        y = max(0, min(height - 1, y))
        x = max(0, min(width - 1, x))
        grid[y][x] = '●'
    
    # Print plot
    print(f"\n{'─' * (width + 2)}")
    for row in grid:
        print(f"│{''.join(row)}│")
    print(f"{'─' * (width + 2)}")
    print(f" E0{' ' * (width - 6)}E{len(values)-1}")
    print(f" Loss: {values[-1]:.4f} (min: {min(values):.4f}, max: {max(values):.4f})")

In [2]:
parent_dir = '/data/garfinkeljb/CloudCode/Noisy_Datasets/'

pth_dir = '/data/garfinkeljb/CloudCode/Models/SUPPORT/pth'       # path to the pth directory (containing models)
tsboard_dir = '/data/garfinkeljb/CloudCode/Models/SUPPORT/tsboard'   # path to the tsboard directory

In [3]:
init_epoch = 0                              # epoch to start training from (need epoch-1 model)
#n_epochs = 75                               # number of epochs of training
n_epochs = 150                               # number of epochs of training

# dataset-dependent parameters
patch_size = [64, 128, 128]                   # size of the patches
#patch_size = [21, 128, 128]                    # size of the patches
train_patch_interval = [1, 64, 64]
#train_patch_interval = [1, 64, 64]
test_patch_interval = [1, 9, 9]
#test_patch_interval = [1, 16, 16]
train_batch_size = 256
batch_size = 128                            # size of the batches
prefetch_factor = 8                         # number of batches to prefetch
bs_size = [3, 3]                            # the size of the blind spot
#bs_size = [1, 1]                            # the size of the blind spot
input_frames = patch_size[0]                # of input frames
is_zarr = False                             # noisy_data is zarr
is_folder = False                           # noisy_data is folder

# network related parameters
lr = 5E-4                                   # adam: learning rate
loss_coef = [0.5, 0.5]                      # L1/L2 loss coefficients
depth = 5                                   # the number of blind spot convolutions, must be an odd number
blind_conv_channels = 64                    # the number of channels of blind spot convolutions
one_by_one_channels = [32, 16]              # the number of channels of 1x1 convolutions
last_layer_channels = [64, 32, 16]          # the number of channels of 1x1 convs after UNet
#bp = True                                   # blind plane
bp = False
unet_channels = [64, 128, 256, 512, 1024]   # the number of channels of UNet
gpu = [0]                                   # GPU to use
n_cpu = 8                                   # number of cpu threads to use during batch generation
use_amp = True                              # Use automatic mixed precision for training
use_CPU = False                             # use CPU
random_seed = 0                             # random seed for rng

# logging
logging_interval_batch = 1E5                # interval between logging info (in batches)
logging_interval = 1                        # interval between logging info (in epochs)
sample_interval = 1e5                       # interval between saving denoised samples
sample_max_t = 1                            # maximum time step of saving sample
checkpoint_interval = 1                     # interval between saving trained models (in epochs)
checkpoint_interval_batch = 1e5             # interval between saving trained models (in batches)

In [4]:
arg_dict = {
    'random_seed': random_seed,
    'epoch': init_epoch,
    'n_epochs': n_epochs,
    'pth_dir': pth_dir,
    'tsboard_dir': tsboard_dir,
    'input_frames': input_frames,
    'cuda_device': gpu,
    'is_zarr': is_zarr,
    'is_folder': is_folder,
    'patch_size': patch_size,
    'patch_interval': train_patch_interval,
    'batch_size': train_batch_size,
    'depth': depth,
    'blind_conv_channels': blind_conv_channels,
    'one_by_one_channels': one_by_one_channels,
    'last_layer_channels': last_layer_channels,
    'bs_size': bs_size,
    'bp': bp,
    'unet_channels': unet_channels,
    'lr': lr,
    'loss_coef': loss_coef,
    'use_amp': use_amp,
    'use_CPU': use_CPU,
    'n_cpu': n_cpu,
    'prefetch_factor': prefetch_factor,
    'logging_interval_batch': logging_interval_batch,
    'logging_interval': logging_interval,
    'sample_interval': sample_interval,
    'sample_max_t': sample_max_t,
    'checkpoint_interval': checkpoint_interval,
    'checkpoint_interval_batch': checkpoint_interval_batch
}


In [5]:
for path in list(Path(parent_dir).glob('*')):
    if '.DS_Store' in str(path): os.remove(path)
# clean_datasets_dir = '/data/garfinkeljb/CloudCode/Clean_Datasets/SUPPORT'
# clean_datasets_dir = list(Path(clean_datasets_dir).glob('*'))
# clean_datasets_dir0 = []
# for path in clean_datasets_dir:
#     if '.log' not in str(path):
#         clean_datasets_dir0.append(path)
# clean_datasets_dir = clean_datasets_dir0
# clean_stems = [str(path.stem) for path in clean_datasets_dir]
# clean_datasets_dir = [str(path) for path in clean_datasets_dir]
# clean_datasets_log_path = '/data/garfinkeljb/CloudCode/Clean_Datasets/SUPPORT/completed.log'
# with open(clean_datasets_log_path, 'r') as log_file:
#     completed = log_file.readlines()
# completed = [a.replace('\n', '') for a in completed]
# for stem in clean_stems:
#     completed.append(stem)
# clean_stems = completed

# set_a = set(clean_stems)

datasets_dir = list(Path(parent_dir).glob('*'))
datasets_stems = [str(path.stem) for path in datasets_dir]
datasets_dir = [str(path) for path in datasets_dir][::-1]
# set_b = set(datasets_stems)
# stems = set_b and set_a
# print(stems)
# remove_dir = []
# for path in datasets_dir:
#     for stem in stems:
#         if stem in path:
#             remove_dir.append(path)
# if len(remove_dir) != 0: 
#     for dir in remove_dir: datasets_dir.remove(dir)
# print(remove_dir)
# print(datasets_dir)

In [6]:
# from pathlib import Path

# dirs_6xfad = list(Path('/data/garfinkeljb/CloudCode/Noisy_Datasets/6xFAD').glob('*'))
# dirs_controls = list(Path('/data/garfinkeljb/CloudCode/Noisy_Datasets/Controls').glob('*'))
# #datasets_dir = dirs_6xfad + dirs_controls
# datasets_dir = dirs_controls

# clean_datasets_log_path = '/data/garfinkeljb/CloudCode/Clean_Datasets/SUPPORT/completed.log'
# with open(clean_datasets_log_path, 'r') as log_file:
#     completed = [line.strip() for line in log_file.readlines()]

# datasets_dir = [str(path) for path in datasets_dir if str(path.stem) not in completed]


# # print("Removed:", [str(path) for path in datasets_dir if str(path.stem) in completed])
# print("Remaining:", datasets_dir)
# print(f"Total in both dirs: {len(datasets_dir)}")
# print(f"Completed: {len(completed)}")
# print(f"Remaining after filter: {len(datasets_dir)}")

In [7]:
def validate(test_dataloader, model):
    """
    Validate a model with a test data
    
    Arguments:
        test_dataloader: (Pytorch DataLoader)
            Should be DatasetFRECTAL_test_stitch!
        model: (Pytorch nn.Module)

    Returns:
        denoised_stack: denoised image stack (Numpy array with dimension [T, X, Y])
    """
    with torch.no_grad():
        model.eval()
        # initialize denoised stack to NaN array.
        denoised_stack = np.zeros(test_dataloader.dataset.noisy_image.shape, dtype=np.float32)
        
        # stitching denoised stack
        # insert the results if the stack value was NaN
        # or, half of the output volume
        for _, (noisy_image, _, single_coordinate) in enumerate(tqdm(test_dataloader, unit='batches', desc="validate")):
            noisy_image = noisy_image.cuda() #[b, z, y, x]
            noisy_image_denoised = model(noisy_image)
            T = noisy_image.size(1)
            for bi in range(noisy_image.size(0)): 
                stack_start_w = int(single_coordinate['stack_start_w'][bi])
                stack_end_w = int(single_coordinate['stack_end_w'][bi])
                patch_start_w = int(single_coordinate['patch_start_w'][bi])
                patch_end_w = int(single_coordinate['patch_end_w'][bi])

                stack_start_h = int(single_coordinate['stack_start_h'][bi])
                stack_end_h = int(single_coordinate['stack_end_h'][bi])
                patch_start_h = int(single_coordinate['patch_start_h'][bi])
                patch_end_h = int(single_coordinate['patch_end_h'][bi])

                stack_start_s = int(single_coordinate['init_s'][bi])
                
                denoised_stack[stack_start_s+(T//2), stack_start_h:stack_end_h, stack_start_w:stack_end_w] \
                    = noisy_image_denoised[bi].squeeze()[patch_start_h:patch_end_h, patch_start_w:patch_end_w].cpu()

        # change nan values to 0 and denormalize
        denoised_stack = denoised_stack * test_dataloader.dataset.std_image.numpy() + test_dataloader.dataset.mean_image.numpy()

        return denoised_stack

In [ ]:
for dir in datasets_dir:
    channel_num = 1
    # if len(list(Path(dir).glob('*'))) > 1:
    #     for path in list(Path(dir).glob('*')):
    #         if '.DS_Store' in str(path):
    #             os.remove(path)
    data_path = dir
    if Path(dir).is_dir(): continue
    model_name = Path(data_path).stem
    print(f'{model_name}    ––––    Size (Z, Y, X) = ', end=' ')
    with tifffile.TiffFile(data_path) as tif: Z, Y, X = tif.series[0].shape
    print(f'({Z}, {Y}, {X})')
    output_dir = f'/data/garfinkeljb/CloudCode/Clean_Datasets/SUPPORT/{model_name}/'
    if not Path(output_dir).exists(): Path(output_dir).mkdir(parents=True)
    logs_path = output_dir + f'train_log.log'
    yaml_path = output_dir + f'train_params.yaml'
    arg_dict['noisy_data'] = data_path
    print(arg_dict['noisy_data'])
    arg_dict['exp_name'] = model_name
    arg_dict['patch_interval'] = train_patch_interval
    arg_dict['is_folder'] = False
    
    opt = SimpleNamespace(**arg_dict)
    print('\033[1;31mTraining parameters -----> \033[0m')
    for key, value in arg_dict.items(): print("{:30s} {:s} ".format(key, str(value)))
    if not Path(opt.pth_dir + f'/{model_name}').exists(): Path(opt.pth_dir + f'/{model_name}').mkdir(parents=True)
    if not Path(opt.tsboard_dir + f'/{model_name}').exists(): Path(opt.tsboard_dir + f'/{model_name}').mkdir(parents=True)

    random.seed(0)
    torch.manual_seed(0)
    cuda = torch.cuda.is_available() and (not opt.use_CPU)
    Tensor = torch.cuda.FloatTensor if cuda else torch.Tensor
    rng = np.random.default_rng(random_seed)
    
    # ################################## #
    # LOGGING & SUMMARIES INITIALIZATION
    # ################################## #
    logging.basicConfig(
        level=logging.INFO,
        filename=logs_path,
        filemode="a", 
        format="%(name)s - %(levelname)s - %(message)s"
    )
    with open(yaml_path, 'w') as f: yaml.dump(arg_dict, f, default_flow_style=True)
    writer = SummaryWriter(opt.tsboard_dir + f'/{model_name}')
    
    # ############## #
    # TRAINING
    # ############## #
    dataloader_train = gen_train_dataloader(
        opt.patch_size, 
        opt.patch_interval, 
        opt.batch_size,
        [data_path], 
        opt,
        opt.is_zarr
    )
    model = SUPPORT(
        in_channels=opt.input_frames, 
        mid_channels=opt.unet_channels, 
        depth=opt.depth,
        blind_conv_channels=opt.blind_conv_channels, 
        one_by_one_channels=opt.one_by_one_channels,
        last_layer_channels=opt.last_layer_channels, 
        bs_size=opt.bs_size, 
        bp=opt.bp
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr)
    if cuda: scaler = torch.amp.GradScaler('cuda', enabled=opt.use_amp)
    else: scaler = torch.amp.GradScaler('cpu', enabled=opt.use_amp)
    if cuda: model = model.cuda()
    
    t0 = time.mktime(time.localtime())
    loss_history = []
    for epoch in range(opt.epoch, opt.n_epochs):
        dataloader_train.dataset.precompute_indices()
        loss_list, loss_list_l1, loss_list_l2 = train(
            dataloader_train, 
            model, optimizer, scaler, 
            rng, writer, epoch, opt
        )

        ts = int(time.mktime(time.localtime()) - t0)
        loss_mean = np.mean(np.array(loss_list))
        loss_mean_l1 = np.mean(np.array(loss_list_l1))
        loss_mean_l2 = np.mean(np.array(loss_list_l2))

        writer.add_scalar("E", epoch+1, epoch+1)
        writer.add_scalar("L1", loss_mean_l1, epoch+1)
        writer.add_scalar("L2", loss_mean_l2, epoch+1)
        writer.add_scalar("L", loss_mean, epoch+1)
        logging.info(f"{epoch:03d}\t{loss_mean:04.4f}\t{loss_mean_l1:04.4f}\t{loss_mean_l2:04.4f}\t{ts:d}")
        loss_history.append(loss_mean)
        ascii_plot(loss_history)
        
        if epoch == n_epochs or epoch == n_epochs-1: torch.save(model.state_dict(), opt.pth_dir + f'/{opt.exp_name}/model{epoch+1:02d}.pth')
        #torch.save(optimizer.state_dict(), opt.pth_dir + f'/{opt.exp_name}/optimizer{epoch+1:02d}.pth')
        #if opt.use_amp: torch.save(scaler.state_dict(), opt.pth_dir + f'/{opt.exp_name}/scaler{epoch+1:02d}.pth')
        if epoch % 5 == 0:
            clear_output()
    # ####### #
    # TESTING
    # ####### #
    torch.cuda.empty_cache()
    arg_dict['patch_interval'] = test_patch_interval
    output_file = output_dir + f'{model_name}_SPRT_E{n_epochs:02d}.tif'
    model_path = pth_dir + f'/{opt.exp_name}/model{n_epochs:02d}.pth'
    print("="*60)
    print("TESTING CONFIGURATION")
    print("="*60)
    print(f"Model: {model_name}")
    print(f"Epoch: {epoch}")
    print(f"Model file: {model_name}")
    print(f"Output file: {Path(output_file).stem}")
    print(f"Patch size: {patch_size}")
    print(f"Patch interval: {test_patch_interval}")
    print(f"Batch size: {batch_size*2}")
    print(f"Blind-spot size: {bs_size}")
    print(f"Blind-plane mode: {bp}")
    print("="*60)
    model = SUPPORT(
        in_channels=opt.input_frames, 
        mid_channels=opt.unet_channels, 
        depth=opt.depth,
        blind_conv_channels=opt.blind_conv_channels, 
        one_by_one_channels=opt.one_by_one_channels,
        last_layer_channels=opt.last_layer_channels, 
        bs_size=opt.bs_size, 
        bp=opt.bp
    ).cuda()
    checkpoint = torch.load(model_path, weights_only=True)
    model.load_state_dict(checkpoint)
    
    noisy_tif = torch.from_numpy(np.pad(skio.imread(data_path).astype(np.float32), (((model.in_channels)//2, (model.in_channels)//2), (0,0), (0,0)), 'reflect')).type(torch.FloatTensor)
    print(noisy_tif.shape, noisy_tif.dtype)
    testset = DatasetSUPPORT_test_stitch(noisy_tif, patch_size=patch_size, patch_interval=test_patch_interval)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size*2, num_workers=4)

    print(f"\nRunning denoising on {path}...")
    denoised_stack = validate(testloader, model)

    crop_margin = (model.in_channels) // 2
    denoised_stack_cropped = denoised_stack[crop_margin:-crop_margin, :, :]

    print(f"\nDenoised stack shape: {denoised_stack_cropped.shape}")
    print(f"Saving to: {output_file}")
        
    skio.imsave(output_file, denoised_stack_cropped, metadata={'axes': 'TYX'}, )
    torch.cuda.empty_cache()
    shutil.move(data_path, data_path.split('Noisy_Datasets')[0] + 'Noisy_Datasets/complete' + data_path.split('Noisy_Datasets')[-1])
    shutil.rmtree('/data/garfinkeljb/CloudCode/Models/SUPPORT/pth' + f'/{opt.exp_name}/')
    shutil.rmtree('/data/garfinkeljb/CloudCode/Models/SUPPORT/tsboard' + f'/{opt.exp_name}/')

Epoch 101:  15%|█▌        | 2/13 [00:02<00:12,  1.16s/batches, Lrun=0.547, L=0.548, L1=0.529, L2=0.566]